In [1]:
import openml
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sslae import SSLAEClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
dataset_name = "abalone"
dataset_openml_version = 2

dataset = openml.datasets.get_dataset(
    dataset_name,
    download_data=True,
    download_qualities=True,
    download_features_meta_data=True,
    version=dataset_openml_version,
)

X, y, _, _ = dataset.get_data(
    dataset_format="dataframe", target=dataset.default_target_attribute
)

### Preprocess the data:
- the categorical columns are one-hot-encoded (standardization of numerical columns happens after the train-val-test split)
- the target is LabelEncoded

In [3]:
# Preprocess the features
# Identify categorical columns
cols_to_encode = X.select_dtypes(include=["object", "category"]).columns.to_list()
numerical_cols = X.drop(columns=cols_to_encode).columns.to_list()

# Fill missing values in categorical columns with the most frequent value (mode)
X[cols_to_encode] = X[cols_to_encode].apply(lambda col: col.fillna(col.mode()[0]))
X[numerical_cols] = X[numerical_cols].apply(lambda col: col.fillna(col.mean()))

# keep track of the original column positions
original_columns = X.columns.tolist()
# one-hot encode categorical features
X = pd.get_dummies(X, columns=cols_to_encode, prefix=cols_to_encode)

# get the newly encoded column names
encoded_cols = [col for col in X.columns if col not in original_columns]

# get the indices of the encoded columns
encoded_col_indices = [X.columns.get_loc(col) for col in encoded_cols]
numerical_col_indices = [X.columns.get_loc(col) for col in numerical_cols]

X = X.astype("float32").to_numpy()

le = LabelEncoder()
y = le.fit_transform(y).astype("float32")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 15% of the training data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

# if there are numerical columns, standardize them
if len(numerical_cols):
    scaler = StandardScaler()

    X_train[:, numerical_col_indices] = scaler.fit_transform(
        X_train[:, numerical_col_indices]
    )

    # Transform the test and validation sets using the same scaler (only for columns to standardize)
    X_test[:, numerical_col_indices] = scaler.transform(
        X_test[:, numerical_col_indices]
    )
    X_val[:, numerical_col_indices] = scaler.transform(X_val[:, numerical_col_indices])

#  10% of the train data labeled
X_train_lab, X_train_unlab, y_train_lab, y_train_unlab = train_test_split(
    X_train, y_train, test_size=0.9, random_state=42
)

In [5]:
sslae = SSLAEClassifier(
    n_intrinsic="auto",
    ssl_weight=0.5,
    batch_size=32,
    learning_rate=1e-3,
    # weight_decay=1e-5,
    # dropout=0.1,
    num_epochs=100,
    stopping_crit=10,
    random_state=18,
    verbose=False,
)

In [6]:
sslae.fit(X_train_lab, X_train_unlab, y_train_lab, X_val, y_val)

y_test_pred = sslae.predict(X_test)
y_test_pred_proba = sslae.predict_proba(X_test)
perf_dict = sslae.score(y_test, y_test_pred, y_test_pred_proba)

Global seed set to 18
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name       | Type             | Params
------------------------------------------------
0 | encoder    | Sequential       | 117   
1 | classifier | Sequential       | 58    
2 | decoder    | Sequential       | 122   
3 | r_loss     | SmoothL1Loss     | 0     
4 | c_loss     | CrossEntropyLoss | 0     
------------------------------------------------
297       Trainable params
0         Non-trainable params
297       Total params
0.001     Total estimated model params size (MB)
`Trainer.fit` stopped: `max_epochs=100` reached.


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.9469653367996216
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.9469653367996216
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


In [7]:
perf_dict

{'accuracy': 0.8038277511961722,
 'auprc_macro': 0.884273165521251,
 'auprc_weighted': 0.884273165521251,
 'auc_ovo': 0.8765131786051571,
 'f1_macro': 0.8037996783038255,
 'f1_weighted': 0.8038165220392335,
 'precision_macro': 0.8038528696149994,
 'precision_weighted': 0.8038421045783591,
 'recall_macro': 0.803783304238331,
 'recall_weighted': 0.8038277511961722,
 'execution_time': 74.63268780708313}